## Make dataset out of Hugginggface dataset

In [ ]:
# Cette cellule sera utilisée plus tard pour calculer le FID score
# Une fois que vous aurez généré des images avec votre modèle

In [ ]:
%pip install --upgrade transformers pillow torch torchvision datasets tqdm accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 760.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.

In [ ]:
!pip install pytorch-fid

In [ ]:
import os
from tqdm import tqdm
from datasets import load_dataset
import json
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

print(f"✓ Tous les modules sont chargés avec succès!")

✓ Tous les modules sont chargés avec succès!


In [ ]:
dataset = load_dataset("dpdl-benchmark/oxford_flowers102")

train_dataset = dataset["train"]
fid_train_dataset = dataset["test"]
fid_test_dataset = dataset["validation"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/418M [00:00<?, ?B/s]

data/test-00000-of-00006.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

data/test-00001-of-00006.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

...

data/test-00005-of-00006.parquet:   0%|          | 0.00/418M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1020 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6149 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1020 [00:00<?, ? examples/s]

In [ ]:
shuffled_test = fid_test_dataset.shuffle(seed=42)

blip_test_dataset = shuffled_test.select(range(len(shuffled_test)//3))
llava_test_dataset = shuffled_test.select(range(len(shuffled_test)//3, 2*len(shuffled_test)//3))
gemini_test_dataset = shuffled_test.select(range(2*len(shuffled_test)//3, len(shuffled_test)))

In [4]:
# Sauvegarder toutes les images dans un dossier pour FID
fid_images_dir = "./datasets/oxford_flowers102_fid_images"
os.makedirs(fid_images_dir, exist_ok=True)

print("Préparation des images pour FID (crop centré 512x512)...")
for idx, item in enumerate(tqdm(fid_train_dataset)):
    image = item["image"]

    # Crop centré 512x512
    width, height = image.size

    # Calculer les coordonnées pour un crop centré
    left = (width - 512) // 2
    top = (height - 512) // 2
    right = left + 512
    bottom = top + 512

    # Si l'image est plus petite que 512x512, on la redimensionne d'abord
    if width < 512 or height < 512:
        # Redimensionner en gardant le ratio, puis crop
        scale = max(512 / width, 512 / height)
        new_width = int(width * scale)
        new_height = int(height * scale)
        image = image.resize((new_width, new_height), Image.LANCZOS)
        width, height = image.size
        left = (width - 512) // 2
        top = (height - 512) // 2
        right = left + 512
        bottom = top + 512

    image_cropped = image.crop((left, top, right, bottom))
    image_cropped.save(f"{fid_images_dir}/{idx}.png")

print(f"✓ {len(fid_train_dataset)} images sauvegardées dans {fid_images_dir}")
print("\nMaintenant, calculons les statistiques FID...")

# Calculer les statistiques FID
!python -m pytorch_fid {fid_images_dir} dummy_path --save-stats

Préparation des images pour FID (crop centré 512x512)...


100%|██████████| 6149/6149 [17:20<00:00,  5.91it/s]


✓ 6149 images sauvegardées dans ./datasets/oxford_flowers102_fid_images

Maintenant, calculons les statistiques FID...
Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:01<00:00, 72.0MB/s]
Saving statistics for ./datasets/oxford_flowers102_fid_images
100% 123/123 [42:14<00:00, 20.61s/it]


In [37]:
# Vérifier que les images FID existent
import os
import shutil
fid_dir = "./datasets/oxford_flowers102_fid_images"
num_images = len([f for f in os.listdir(fid_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
print(f"✓ {num_images} images trouvées dans {fid_dir}")

if num_images > 0:
    # Calculer les statistiques FID sur le dataset réel (test set)
    print("\nCalcul des statistiques FID sur le dataset réel...")

    # Créer le dossier datasets s'il n'existe pas
    os.makedirs("./datasets", exist_ok=True)

    # Supprimer les anciens fichiers s'ils existent
    generated_file = f"{fid_dir}.npz"
    target_file = "./datasets/stats_oxford_flowers.npz"

    if os.path.exists(generated_file):
        os.remove(generated_file)
        print(f"Ancien fichier supprimé : {generated_file}")

    if os.path.exists(target_file):
        os.remove(target_file)
        print(f"Ancien fichier supprimé : {target_file}")

    # Calculer les statistiques avec un dossier temporaire comme 2e argument
    # pytorch_fid va créer le fichier .npz pour le premier chemin
    !python -m pytorch_fid {fid_dir} ./temp_dummy_folder --save-stats

    # Déplacer le fichier généré vers le bon emplacement
    if os.path.exists(generated_file):
        shutil.move(generated_file, target_file)
        print(f"\n✓ Statistiques sauvegardées dans {target_file}")

        # Vérifier que le fichier est valide
        import numpy as np
        try:
            data = np.load(target_file)
            print(f"✓ Fichier valide : contient {list(data.keys())}")
            data.close()
        except Exception as e:
            print(f"❌ Fichier invalide : {e}")
    else:
        print(f"\n❌ Erreur : le fichier {generated_file} n'a pas été créé")
else:
    print(f"❌ Aucune image trouvée dans {fid_dir}")

✓ 6149 images trouvées dans ./datasets/oxford_flowers102_fid_images

Calcul des statistiques FID sur le dataset réel...
Saving statistics for ./datasets/oxford_flowers102_fid_images
100% 123/123 [44:42<00:00, 21.81s/it]

❌ Erreur : le fichier ./datasets/oxford_flowers102_fid_images.npz n'a pas été créé


# Pas toucher

In [ ]:
from tqdm import tqdm
# Sauvegarder toutes les images dans un dossier pour FID
fid_images_dir = "./datasets/oxford_flowers102_fiiid"
os.makedirs(fid_images_dir, exist_ok=True)

print("Préparation des images pour FID (crop centré 512x512)...")
for idx, item in enumerate(tqdm(fid_test_dataset)):
    image = item["image"]

    # Crop centré 512x512
    width, height = image.size

    # Calculer les coordonnées pour un crop centré
    left = (width - 512) // 2
    top = (height - 512) // 2
    right = left + 512
    bottom = top + 512

    # Si l'image est plus petite que 512x512, on la redimensionne d'abord
    if width < 512 or height < 512:
        # Redimensionner en gardant le ratio, puis crop
        scale = max(512 / width, 512 / height)
        new_width = int(width * scale)
        new_height = int(height * scale)
        image = image.resize((new_width, new_height), Image.LANCZOS)
        width, height = image.size
        left = (width - 512) // 2
        top = (height - 512) // 2
        right = left + 512
        bottom = top + 512

    image_cropped = image.crop((left, top, right, bottom))
    image_cropped.save(f"{fid_images_dir}/{idx}.png")

print(f"✓ {len(fid_train_dataset)} images sauvegardées dans {fid_images_dir}")
print("\nMaintenant, calculons les statistiques FID...")


Préparation des images pour FID (crop centré 512x512)...


100%|██████████| 1020/1020 [02:56<00:00,  5.77it/s]

✓ 6149 images sauvegardées dans ./datasets/oxford_flowers102_fiiid

Maintenant, calculons les statistiques FID...


In [ ]:
!python -m pytorch_fid datasets/oxford_flowers102_fiiid datasets/stats_oxford_flowers.npz

100% 21/21 [07:18<00:00, 20.86s/it]
FID:  17.46993076898383


# Import fichier

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!ls '/content/drive/My Drive/dla'

generated_images


In [10]:
import shutil
import os

source_path = '/content/drive/My Drive/dla/generated_images'
destination_path = './generated_images' # Cela le copiera dans le répertoire de travail actuel

if os.path.exists(source_path):
    # Si le dossier de destination existe déjà, on le supprime d'abord pour s'assurer d'avoir la dernière version
    if os.path.exists(destination_path):
        shutil.rmtree(destination_path)
        print(f"Ancien dossier '{destination_path}' supprimé.")

    shutil.copytree(source_path, destination_path)
    print(f"Dossier '{source_path}' copié vers '{destination_path}'.")
else:
    print(f"Le dossier source '{source_path}' n'existe pas. Veuillez vérifier le chemin sur votre Google Drive.")


Dossier '/content/drive/My Drive/dla/generated_images' copié vers './generated_images'.


# Creation dossier avec toute les images

In [38]:
import os
import shutil
from tqdm import tqdm

# Chemins des trois dossiers source (MODIFIE CES CHEMINS)
dossier1 = "./generated_images/base/blip_captions"
dossier2 = "./generated_images/base/llava_captions"  # Exemple
dossier3 = "./generated_images/base/gemini_captions"  # Exemple

# Dossier de destination
dossier_concat = "./generated_images/base/all"
os.makedirs(dossier_concat, exist_ok=True)

print("Concaténation des images de 3 dossiers avec renommage automatique...\n")

# Compteur global pour les images
compteur_global = 0

# Liste des dossiers source
dossiers_source = [dossier1, dossier2, dossier3]
noms_dossiers = ["blip", "llava", "gemini"]  # Préfixes pour identifier l'origine

for idx, dossier_src in enumerate(dossiers_source):
    if not os.path.exists(dossier_src):
        print(f"⚠️  Le dossier '{dossier_src}' n'existe pas, ignoré.")
        continue

    # Lister toutes les images du dossier
    images = [f for f in os.listdir(dossier_src) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    print(f"📁 Traitement du dossier '{dossier_src}' : {len(images)} images trouvées")

    for image_name in tqdm(images, desc=f"Copie depuis {noms_dossiers[idx]}"):
        # Chemin source
        src_path = os.path.join(dossier_src, image_name)

        # Extension de l'image
        extension = os.path.splitext(image_name)[1]

        # Nouveau nom avec préfixe et compteur global
        nouveau_nom = f"{noms_dossiers[idx]}_{compteur_global:05d}{extension}"

        # Chemin destination
        dst_path = os.path.join(dossier_concat, nouveau_nom)

        # Copier l'image
        shutil.copy2(src_path, dst_path)

        compteur_global += 1

# Vérification finale
total_images = len([f for f in os.listdir(dossier_concat) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

print(f"\n✅ Concaténation terminée!")
print(f"📊 Total d'images dans '{dossier_concat}' : {total_images}")
print(f"📍 Chemin absolu : {os.path.abspath(dossier_concat)}")

Concaténation des images de 3 dossiers avec renommage automatique...

📁 Traitement du dossier './generated_images/base/blip_captions' : 340 images trouvées


Copie depuis blip: 100%|██████████| 340/340 [00:01<00:00, 278.92it/s]


📁 Traitement du dossier './generated_images/base/llava_captions' : 340 images trouvées


Copie depuis llava: 100%|██████████| 340/340 [00:01<00:00, 287.51it/s]


📁 Traitement du dossier './generated_images/base/gemini_captions' : 340 images trouvées


Copie depuis gemini: 100%|██████████| 340/340 [00:04<00:00, 71.30it/s] 


✅ Concaténation terminée!
📊 Total d'images dans './generated_images/base/all' : 1020
📍 Chemin absolu : /content/generated_images/base/all


# FID Base

### Blip

In [55]:
!python -m pytorch_fid generated_images/base/blip_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:23<00:00, 20.52s/it]
FID:  123.71027138855291


### Gemini

In [56]:
!python -m pytorch_fid generated_images/base/gemini_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:23<00:00, 20.46s/it]
FID:  97.94462894731441


### Llava

In [57]:
!python -m pytorch_fid generated_images/base/Llava_captions datasets/stats_oxford_flowers.npz


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pytorch_fid/__main__.py", line 3, in <module>
    pytorch_fid.fid_score.main()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_fid/fid_score.py", line 313, in main
    fid_value = calculate_fid_given_paths(args.path,
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_fid/fid_score.py", line 253, in calculate_fid_given_paths
    raise RuntimeError('Invalid path: %s' % p)
RuntimeError: Invalid path: generated_images/base/Llava_captions


### All

In [58]:
!python -m pytorch_fid generated_images/base/all datasets/stats_oxford_flowers.npz


100% 21/21 [07:13<00:00, 20.63s/it]
FID:  92.26691895820329


# FID Blip


## Blip

In [59]:
!python -m pytorch_fid generated_images/blip/blip_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:22<00:00, 20.35s/it]
FID:  51.27626853773188


## Gemini

In [60]:
!python -m pytorch_fid generated_images/blip/gemini_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:22<00:00, 20.36s/it]
FID:  54.46268636816524


## Llava

In [61]:
!python -m pytorch_fid generated_images/blip/llava_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:26<00:00, 20.95s/it]
FID:  67.26344008410513


## All

In [62]:
!python -m pytorch_fid generated_images/blip/all datasets/stats_oxford_flowers.npz


100% 21/21 [07:06<00:00, 20.29s/it]
FID:  32.627843190341366


# FID Gemini

## Blip

In [63]:
!python -m pytorch_fid generated_images/gemini/blip_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:31<00:00, 21.67s/it]
FID:  67.54707128437434


## Gemini

In [64]:
!python -m pytorch_fid generated_images/gemini/gemini_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:23<00:00, 20.49s/it]
FID:  53.81970238853873


## Llava

In [65]:
!python -m pytorch_fid generated_images/gemini/llava_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:22<00:00, 20.41s/it]
FID:  62.10955733773724


## All

In [66]:
!python -m pytorch_fid generated_images/gemini/all datasets/stats_oxford_flowers.npz


100% 21/21 [07:14<00:00, 20.71s/it]
FID:  35.90965395181041


# FID Lllava

## Blip

In [67]:
!python -m pytorch_fid generated_images/llava/blip_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:24<00:00, 20.60s/it]
FID:  63.16206190325278


## Gemini

In [68]:
!python -m pytorch_fid generated_images/llava/gemini_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:28<00:00, 21.24s/it]
FID:  70.2920469627174


## Llava

In [69]:
!python -m pytorch_fid generated_images/llava/gemini_captions datasets/stats_oxford_flowers.npz


100% 7/7 [02:21<00:00, 20.23s/it]
FID:  70.2920469627174


## All

In [70]:
!python -m pytorch_fid generated_images/llava/all datasets/stats_oxford_flowers.npz


100% 21/21 [07:10<00:00, 20.51s/it]
FID:  37.42790716019351
